Environment Setup

In [ ]:
!nvidia-smi

Mon Jun 29 11:52:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip -q install \
transformers==4.53.2 \
peft==0.16.0 \
trl==0.19.1 \
accelerate==1.8.1 \
bitsandbytes==0.46.1 \
datasets==3.6.0 \
huggingface_hub==0.33.0 \
sentencepiece \
safetensors

In [ ]:
import torch
import transformers
import peft
import trl
import bitsandbytes as bnb
import accelerate
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("BNB:", bnb.__version__)
print("Accelerate:", accelerate.__version__)

Torch: 2.11.0+cu128
Transformers: 4.53.2
PEFT: 0.16.0
TRL: 0.19.1
BNB: 0.46.1
Accelerate: 1.8.1


In [ ]:
from huggingface_hub import notebook_login
notebook_login()
from huggingface_hub import whoami
print(whoami())

In [ ]:
import torch
from transformers import (
   AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import(
    prepare_model_for_kbit_training,
    LoraConfig,
    get_peft_model
)
from trl import SFTTrainer,SFTConfig
from datasets import load_dataset

Model Loading

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
model_name = "Qwen/Qwen2.5-3B-Instruct"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
model = prepare_model_for_kbit_training(model)

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
from peft import get_peft_model

model = get_peft_model(
    model,
    lora_config
)
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


Dataset

In [ ]:
dataset = load_dataset(
    "iamtarun/python_code_instructions_18k_alpaca",
    split="train"
)

print(dataset)

In [ ]:
def formatting_func(example):
    return {
        "text": f"""<|im_start|>system
You are an expert software engineer.
<|im_end|>

<|im_start|>user
{example["instruction"]}

{example["input"]}
<|im_end|>

<|im_start|>assistant
{example["output"]}
<|im_end|>"""
    }

dataset = dataset.map(formatting_func)

In [ ]:
print(dataset["text"][0])

<|im_start|>system
You are an expert software engineer.
<|im_end|>

<|im_start|>user
Create a function to calculate the sum of a sequence of integers.

[1, 2, 3, 4, 5]
<|im_end|>

<|im_start|>assistant
# Python code
def sum_sequence(sequence):
  sum = 0
  for num in sequence:
    sum += num
  return sum
<|im_end|>


Training

In [ ]:
training_args = SFTConfig(
    output_dir="./codeforge-qwen",
    dataset_text_field="text",
    max_steps=100,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    max_seq_length=512,
    packing=False,
    bf16=True,
    report_to="none",
)

In [ ]:
import inspect
from trl import SFTTrainer
print(inspect.signature(SFTTrainer.__init__))

(self, model: Union[str, torch.nn.modules.module.Module, transformers.modeling_utils.PreTrainedModel], args: Union[trl.trainer.sft_config.SFTConfig, transformers.training_args.TrainingArguments, NoneType] = None, data_collator: Optional[transformers.data.data_collator.DataCollator] = None, train_dataset: Union[datasets.arrow_dataset.Dataset, datasets.iterable_dataset.IterableDataset, NoneType] = None, eval_dataset: Union[datasets.arrow_dataset.Dataset, dict[str, datasets.arrow_dataset.Dataset], NoneType] = None, processing_class: Union[transformers.tokenization_utils_base.PreTrainedTokenizerBase, transformers.image_processing_utils.BaseImageProcessor, transformers.feature_extraction_utils.FeatureExtractionMixin, transformers.processing_utils.ProcessorMixin, NoneType] = None, compute_loss_func: Optional[Callable] = None, compute_metrics: Optional[Callable[[transformers.trainer_utils.EvalPrediction], dict]] = None, callbacks: Optional[list[transformers.trainer_callback.TrainerCallback]] 

In [ ]:
import inspect
from trl import SFTConfig
print(inspect.signature(SFTConfig))

(output_dir: Optional[str] = None, overwrite_output_dir: bool = False, do_train: bool = False, do_eval: bool = False, do_predict: bool = False, eval_strategy: Union[transformers.trainer_utils.IntervalStrategy, str] = 'no', prediction_loss_only: bool = False, per_device_train_batch_size: int = 8, per_device_eval_batch_size: int = 8, per_gpu_train_batch_size: Optional[int] = None, per_gpu_eval_batch_size: Optional[int] = None, gradient_accumulation_steps: int = 1, eval_accumulation_steps: Optional[int] = None, eval_delay: Optional[float] = 0, torch_empty_cache_steps: Optional[int] = None, learning_rate: float = 2e-05, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, max_grad_norm: float = 1.0, num_train_epochs: float = 3.0, max_steps: int = -1, lr_scheduler_type: Union[transformers.trainer_utils.SchedulerType, str] = 'linear', lr_scheduler_kwargs: Union[dict[str, Any], str, NoneType] = <factory>, warmup_ratio: float = 0.0, warmup

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args,
)

In [ ]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.094000
20,0.677700
30,0.605100
40,0.635500
50,0.600000
60,0.654200


Step,Training Loss
10,1.094000
20,0.677700
30,0.605100
40,0.635500
50,0.600000
60,0.654200
70,0.815800
80,0.702100
90,0.608700
100,0.653700


TrainOutput(global_step=100, training_loss=0.7046845817565918, metrics={'train_runtime': 1851.8753, 'train_samples_per_second': 0.432, 'train_steps_per_second': 0.054, 'total_flos': 2658929859919872.0, 'train_loss': 0.7046845817565918})

In [ ]:
trainer.save_model("./codeforge_qwen_lora")
tokenizer.save_pretrained("./codeforge_qwen_lora")

('./codeforge_qwen_lora/tokenizer_config.json',
 './codeforge_qwen_lora/special_tokens_map.json',
 './codeforge_qwen_lora/chat_template.jinja',
 './codeforge_qwen_lora/vocab.json',
 './codeforge_qwen_lora/merges.txt',
 './codeforge_qwen_lora/added_tokens.json',
 './codeforge_qwen_lora/tokenizer.json')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16,
)
model = PeftModel.from_pretrained(
    base_model,
    "./codeforge_qwen_lora"
)
tokenizer = AutoTokenizer.from_pretrained("./codeforge_qwen_lora")

Fine-Tuned Model Output

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(
    base_model,
    "./codeforge_qwen_lora"
)
tokenizer = AutoTokenizer.from_pretrained("./codeforge_qwen_lora")
prompt = """Write a Python function to check whether a string is a palindrome.
Explain the code."""
messages = [
    {
        "role": "user",
        "content": prompt
    }
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.2,
        do_sample=True,
    )
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True,
)

print(response)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a Python function to check whether a string is a palindrome.
Explain the code.
assistant
def is_palindrome(string):
    # Convert the string to lowercase and remove all non-alphanumeric characters
    string = ''.join([char.lower() for char in string if char.isalnum()])
    
    # Check if the string is equal to its reverse
    return string == string[::-1]

# Example usage:
print(is_palindrome("A man, a plan, a canal: Panama")) # True
print(is_palindrome("race a car")) # False


In [ ]:
odel.push_to_hub("____")
tokenizer.push_to_hub("___-")

Base Model Output

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

prompt = """Write a Python function to check whether a string is a palindrome.
Explain the code."""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(base_model.device)

with torch.no_grad():
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.2,
        do_sample=True,
    )

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True,
)

print(response)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a Python function to check whether a string is a palindrome.
Explain the code.
assistant
Certainly! A palindrome is a word, phrase, number, or other sequences of characters which reads the same backward as forward, ignoring spaces, punctuation, and capitalization.

Here's a Python function to check if a given string is a palindrome:

```python
def is_palindrome(s: str) -> bool:
    """
    Check if the input string is a palindrome.

    Parameters:
    s (str): The string to check.

    Returns:
    bool: True if the string is a palindrome, False otherwise.
    """
    
    # Normalize the string: remove non-alphanumeric characters and convert to lowercase
    normalized_str = ''.join(char.lower() for char in s if char.isalnum())
    
    # Compare the normalized string with its reverse
    return normalized_str == normalized_str[::-1]
```

### Explanation of the Code

1. **Function Definition**:
   